# பாடம் 13 - கோங்கி அறிவுக் கிராப்களுடன் முகவர் நினைவகம்


## அமைப்பு

இந்த நோட்புக் எடுப்புக்கள் பயன்படுத்தி [**Cognee**](https://www.cognee.ai/) அறிவுக் கிராமங்கள் மற்றும் **Microsoft Agent Framework** (MAF) மூலம் நிலையான நினைவகத்துடன் ஒரு புத்திசாலி **கோடிங் உதவியாளர்** உருவாக்குவதை விளக்குகிறது.

Cognee அமைக்கப்படாத உரையை ஒரு அமைந்த, கேள்வி கேட்கக்கூடிய அறிவுக் கிராமமாக மாற்றுகிறது, இதில் வெக்டர் எம்பெடிங்களால் ஆதரிக்கப்படுகிறது — உங்கள் முகவரிக்கு வளமான, தொடர்பு-அறிந்த நீண்டகால நினைவகத்தை வழங்குகிறது.

### நீங்கள் கற்றுக்கொள்ளப்போகும் விஷயங்கள்
1. **அறிவுக் கிராமங்களை உருவாக்குக**: அபிவிருத்தி profயில்களை மற்றும் சிறந்த நடைமுறைகளை அமைந்த, கேள்வி கேட்கக்கூடிய அறிவாக மாற்றுக.
2. **Cognee-வை MAF உடன் இணைக்கவும்**: `@tool` செயல்பாடுகளைப் பயன்படுத்தி MAF முகவர் Cognee-வின் அறிவுக் கிராமத்தை கேள்வி செய்ய அனுமதிக்கவும்.
3. **அமர்வுத் தெரிவு உறவுகள்**: ஒரே அமர்வில் பல கேள்விகளுக்கு இடையே சூழலை பராமரிக்கவும்.
4. **நீண்டகால நினைவகம்**: முக்கிய அறிவை அமர்வுகளில் நிலைத்துவைத்துக் கொண்டு புதிய உரையாடல்களில் மீட்டெடுக்கவும்.

### முன் தேவைகள்
- Python 3.9+
- அமர்வு மேலாண்மைக்காக உள்ளூர் ரெடிஸ் (Redis) இயங்குகிறது (`docker run -d -p 6379:6379 redis`)
- ஒரு LLM API விசை (உதா., OpenAI) — `.env` இல் `LLM_API_KEY` அமைக்கவும்
- `.env` இல் `CACHING=true` (Cognee அமர்வுகளுக்கு தேவையாகும்)
- Azure AI Foundry திட்டம் ஒரு வடிவமைக்கப்பட்ட உரையாடல் மாதிரியுடன்
- `.env` இல் `AZURE_AI_PROJECT_ENDPOINT` மற்றும் `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI அங்கீகாரம் (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## முகவர் நினைவகத்தின் வகைகள்

இந்த நோட்புக், முக்கிய பாடம் 13 நோட்புக்கில் உள்ள மூன்று நினைவக வகைகளைப் பயன்படுத்துகிறது, ஆனால் Cognee ஐ நீண்டகால நினைவகம் பின்புலமாகக் கொண்டுள்ளது:

| நினைவக வகை | இயந்திரம் | கால அளவு |
|---|---|---|
| **செயல்பாட்டு** | `agent.create_session()` (MAF) | ஒரே உரையாடல் |
| **குறுகியகால** | Cognee அமர்வு கேஷ் (Redis) | ஒரே அமர்வு |
| **நீண்டகால** | Cognee அறிவு வரைபடம் + வெக்டர்கள் | நிரந்தரம் |

### Cognee நினைவக கட்டமைப்பு
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## காக்னி சேமிப்பை தயார் செய்க


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## பகுதி 1 — அறிவுத்தளத்தை உருவாக்குதல்

எங்கள் கோடிங் உதவியாளருக்கான முழுமையான அறிவுத்தளத்தை உருவாக்க நாம் மூன்று வகையான தரவுகளை ஏற்றுக் கொள்கிறோம்:

1. **உரையாளர் சுயவிவரம்** — தனிப்பட்ட நுணுக்கமும் தொழில்நுட்ப பின்னணியும்  
2. **Python சிறந்த நடைமுறைகள்** — நடைமுறைகள் உடன் Python யின் சென்ஸ்  
3. **வரலாற்றுப் பேச்சுவார்த்தைகள்** — உரைநடை மற்றும் AI உதவியாளர்களுக்கிடையேயான கடந்த கேள்வி-பதில் அமர்வுகள்


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## அறிவுக் குழாய் வரைபடத்தை காட்சிப்படுத்துக

Cognee களவியல் மற்றும் உறவுகளை பிரிந்தெடுத்துள்ள இடமிருந்து ஒரு செயல்பாட்டுடைய HTML காட்சிப்படுத்தலை உருவாக்க முடியும்.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify மூலம் நினைவகத்தை வளப்படுத்துங்கள்

`memify()` அறிவியல் வரைபடத்தை பகுக்கும் மற்றும் புத்திசாலி விதிகளை உருவாக்குகிறது — வடிவங்களை, சிறந்த நடைமுறைகளை மற்றும் கருத்துக்களின் இடையேயான தொடர்புகளை அடையாளம் கண்டு.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## பகுதி 2 — Cognee கருவிகளுடன் MAF முகவர்

இதோடு நாம் `@tool` செயல்பாடுகளைப் பயன்படுத்தி Cognee-வின் அறிவுக் கிராப்-ஐ கேள்வி செய்யக்கூடிய MAF முகவரை உருவாக்குகிறோம். இது முகவருக்கு உரையாடல் சூழ்நிலையை பராமரிக்கwhile குரல் உணர்வுக் கண்காணிப்பு semantic தேடலில் முழு சக்தியையும் பெறுமதிக்கூறும்.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## அமர்வு உடன் பணியாற்றும் நினைவகம்

`AgentSession` (`agent.create_session()` மூலம் உருவாக்கப்பட்டது) ஒரு அமர்வில் பணியாற்றும் நினைவகத்தை வழங்குகிறது. முகவர் முன்னைய செய்திகளுக்கு திரும்பி பார்க்கவும், Cognee-ன் நீண்டகால அறிவு வரைபடத்தை விசாரணை செய்யவும் முடியும்.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## புதிய அமர்வு — நீண்ட கால நினைவாற்றல் நிலைத்திருக்கும்

புதிய அமர்வைத் தொடங்குவது செயல்பாட்டு நினைவேதத்தை அழிக்கும், ஆனால் Cognee அறிவு வரைபடம் இன்னும் கிடைக்கும். முகவர் ஒரே நீண்டகால அறிவை முற்றிலும் புதிய உரையாடலில் மீட்டெடுக்கலாம்.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## சுருக்கம்

இந்த நோட்புக்கில் நீங்கள் **MAF இன் வேலை நினைவகத்தை** (`agent.create_session()`) **Cognee இன் நீண்டகால அறிவுக் காட்சி** உடன் இணைக்கக் கூடிய ஒரு கோடிங் உதவியாளரை உருவாக்கியுள்ளீர்கள்.

### நீங்கள் கற்றுக்கொண்டவை
1. **அறிவுக் காட்சி கட்டமைப்பு**: Cognee அமைப்பற்ற உரையைப் பயன்படுத்தி ஒரு காட்சி + வெக்ட்டர் நினைவகத்தை கட்டுகிறது.
2. **memify உடன் காட்சி வளமேம்பாடு**: உங்கள் நிலையான காட்சியில் இருந்து பெறப்பட்ட உண்மைகள் மற்றும் வளமான உறவுகளை சேர்க்கிறது.
3. **MAF + Cognee ஒருங்கிணைவு**: `@tool` செயல்பாடுகள் ஒரு MAF முகவர் Cognee இன் காட்சியை இயல்புவழியில் விசாரிக்க அனுமதிக்கிறது.
4. **வேலை நினைவகம் + நீண்டகால நினைவகம்**: `AgentSession` (`agent.create_session()` மூலம்) அமர்வு சூழலை வழங்குகிறது, Cognee நிலையான அறிவினை வழங்குகிறது.
5. **NodeSets உடன் வடிகட்டப்பட்ட தேடல்**: அறிவுக் காட்சியின் குறிப்பிட்ட துணுக்குகளை (எ.கா. கொள்கைகள் மட்டும்) இலக்காக்கிறது.

### முக்கிய எடுத்துக்கொள்வுகள்
- **Cognee** அசல் உரையை கட்டமைக்கப்பட்ட, உறவும் அறிந்த நினைவகமாக மாற்றுகிறது — ஒரே நேரத்தில் வெக்ட்டர் நினைவகத்தைவிட சக்திவாய்ந்தது.
- **`@tool` செயல்பாடுகள்** MAF முகவர்கள் மற்றும் வெளிப்புற அறிவு அமைப்புகளுக்கு இடையே தெளிவான பாலமாக உள்ளது.
- **`AgentSession`** (`agent.create_session()` மூலம்) ஒவ்வொரு உரையாடல் சூழலையும் நீண்டகால அறிவிலிருந்து தனிப்படுத்தி வைத்திருக்கிறது.
- ஒரே அறிவுக் காட்சி பல அமர்வுகளுக்கும் முகவர்களுக்கும் சேவை செய்கிறது.

### நிஜ உலக பயன்பாடுகள்
- **ஆ발ோகரர்கள் கூட்டாளிகள்**: குறியீட்டு மதிப்பீடு, சம்பவ பகுப்பாய்வு, கட்டமைப்பு உதவியாளர்கள்
- **வாடிக்கையாளர் முகாமையாளர் கூட்டாளிகள்**: தயாரிப்பு ஆவணங்கள், அடிக்கடி கேட்கப்படும் கேள்விகள் மற்றும் CRM குறிப்புகள் மூலம் ஆதரவு முகவர்கள்
- **கூடுகளில் உள்ள நிபுணர் கூட்டாளிகள்**: கொள்கைகள், சட்டம் அல்லது பாதுகாப்பு உதவியாளர்கள் மார்கிசார்ந்த வழிகாட்டல்களை ஆய்வு செய்கின்றனர்
- **ஒற்றைந்த தரவு அடுக்குகள்**: கட்டமைக்கப்பட்ட மற்றும் அமைப்பற்ற தரவுகளை ஒரே கேள்விக்குரிய காட்சியாக சேர்க்குதல்

### அடுத்த படிகள்
- Cognee இல் காலத்துடன் கூடிய உணர்வை பரிசோதனை செய்யவும்
- துறையனுசரணைகளுக்கு OWL ஒட்டோனாலஜியை வரையறுக்கவும்
- தேடல் மேம்பாட்டிற்கு பயனர் கருத்துப்பரவலைச் சேர்க்கவும்
- ஒரே Cognee நினைவக அடுக்கை பகிரும் பல முகவர் அமைப்புக்களை அளவிடவும்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**அறிவுறுத்தல்**:  
இந்த ஆவணம் AI மொழி பெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயலுகிறோம் எனினும், தானியங்கி மொழிபெயர்ப்பில் தவறுகள் அல்லது மோசமான தமிழாக்கங்கள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனிக்கவும். தாய்மொழியில் உள்ள அசல் ஆவணம் அதிகாரப்பூர்வமான ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழிற்முறை மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பின் பயன்பாட்டால் ஏற்படக்கூடிய தவறறிதல் அல்லது தவறான விளக்கங்களுக்கு எங்களுக்கு பொறுப்பேற்றிருக்கமாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
